In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
# ==========================================================
# ACTIVIDAD 11. Métricas de cada acción para elegir la muestra
# Validez de los datos (periodo completo) y liquidez, precio y
# volatilidad (SOLO periodo de entrenamiento). No excluye nada.
# ==========================================================
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

FIN_ENTRENAMIENTO = pd.Timestamp("2022-12-30")     # propuesta pendiente de aprobación
PRECIO_MIN = 5                  # dólares: umbral habitual para excluir acciones de muy bajo precio
VOLUMEN_DOLARES_MIN = 1_000_000 # dólares negociados al día (mediana)
DIAS_SIN_VOLUMEN_MAX = 0.01     # como máximo el 1 % de las sesiones sin negociación

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ----------------------------------------------------------
# 1. Cargar el DataFrame consolidado (Actividad 10)
# ----------------------------------------------------------
precios = pd.read_parquet(CARPETA_DATOS / "precios_universo.parquet")
universo = pd.read_csv(CARPETA / "universo_acciones.csv", dtype=str, keep_default_na=False)
print("Filas:", len(precios), "| tickers:", precios["ticker"].nunique())

# ----------------------------------------------------------
# 2. Validez de los datos en TODO el periodo (valores imposibles)
# ----------------------------------------------------------
COLS = ["open", "high", "low", "close", "adj_close"]
no_positivo = precios[COLS] <= 0
afectadas = no_positivo.any(axis=1)
print("\n=== PRECIOS NO POSITIVOS ===")
print(no_positivo[afectadas].groupby(precios.loc[afectadas, "ticker"], observed=True).sum())
print(precios[afectadas].groupby("ticker", observed=True)["date"].agg(["min", "max"]))

# Cociente adj_close / close: refleja el ajuste por dividendos y debería estar entre 0 y 1
cociente = precios["adj_close"] / precios["close"]
validez = pd.DataFrame({
    "n_no_positivo": afectadas.groupby(precios["ticker"], observed=True).sum(),
    "cociente_min": cociente.groupby(precios["ticker"], observed=True).min(),
    "cociente_max": cociente.groupby(precios["ticker"], observed=True).max(),
})
print("\n=== COCIENTE adj_close / close ===")
print(validez[["cociente_min", "cociente_max"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("\nLos 10 cocientes mínimos más bajos:")
print(validez.sort_values("cociente_min")[["cociente_min", "cociente_max"]].head(10))

# ----------------------------------------------------------
# 3. Métricas de selección SOLO con el periodo de entrenamiento
# ----------------------------------------------------------
ent = precios[precios["date"] <= FIN_ENTRENAMIENTO].copy()
ajustado = ent["adj_close"].where(ent["adj_close"] > 0)          # evita logaritmos de valores <= 0
por_ticker = ent["ticker"]
ent["rend_log"] = np.log(ajustado).groupby(por_ticker, observed=True).diff()
anterior = ajustado.groupby(por_ticker, observed=True).shift(1)    # precio del día previo
ent["rend_abs"] = (ajustado / anterior - 1).abs()
ent["dolares"] = ent["close"] * ent["volume"]
ent["sin_volumen"] = ent["volume"] == 0

metricas = ent.groupby("ticker", observed=True).agg(
    sesiones=("date", "size"),
    precio_mediano=("close", "median"),
    volumen_dolares_mediano=("dolares", "median"),
    pct_sin_volumen=("sin_volumen", "mean"),
    volatilidad_diaria=("rend_log", "std"),
    max_rend_abs_entr=("rend_abs", "max"),
)
metricas["volatilidad_anual"] = metricas["volatilidad_diaria"] * np.sqrt(252)
metricas = metricas.join(validez)
metricas.index = metricas.index.astype(str)
metricas = metricas.join(universo.set_index("ticker")[["cik", "division_sic", "clases_emisor"]])

print("\n=== ENTRENAMIENTO: de", ent["date"].min().date(), "a", ent["date"].max().date(),
      "|", metricas["sesiones"].max(), "sesiones ===")
resumen = metricas[["precio_mediano", "volumen_dolares_mediano", "pct_sin_volumen", "volatilidad_anual"]].copy()
resumen["volumen_dolares_mediano"] /= 1e6                        # en millones de dólares
print(resumen.describe(percentiles=[0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))

# ----------------------------------------------------------
# 4. Embudo con los umbrales propuestos (cada paso se suma al anterior)
# ----------------------------------------------------------
pasos = {
    "1. Universo": pd.Series(True, index=metricas.index),
    "2. Sin precios no positivos y cociente > 0": (metricas["n_no_positivo"] == 0) & (metricas["cociente_min"] > 0),
    f"3. Precio mediano >= {PRECIO_MIN} $": metricas["precio_mediano"] >= PRECIO_MIN,
    f"4. Volumen mediano >= {VOLUMEN_DOLARES_MIN / 1e6:.0f} M$": metricas["volumen_dolares_mediano"] >= VOLUMEN_DOLARES_MIN,
    f"5. Sesiones sin volumen <= {DIAS_SIN_VOLUMEN_MAX:.0%}": metricas["pct_sin_volumen"] <= DIAS_SIN_VOLUMEN_MAX,
}
acumulado = pd.Series(True, index=metricas.index)
print("\n=== EMBUDO (tickers | empresas) ===")
for nombre, condicion in pasos.items():
    acumulado &= condicion
    print(f"{nombre:45s} {acumulado.sum():5d} | {metricas.loc[acumulado, 'cik'].nunique():5d}")
metricas["pasa_filtros"] = acumulado

# Una sola clase por empresa: la más líquida en el entrenamiento
elegidas = (metricas[metricas["pasa_filtros"]]
            .sort_values("volumen_dolares_mediano", ascending=False)
            .drop_duplicates("cik"))
metricas["elegible"] = metricas.index.isin(elegidas.index)
print(f"{'6. Una clase por empresa (la más líquida)':45s} {len(elegidas):5d} | {elegidas['cik'].nunique():5d}")

print("\nSensibilidad: tickers que pasan los pasos 2 y 5 según precio y volumen mínimos")
base = pasos["2. Sin precios no positivos y cociente > 0"] & pasos[f"5. Sesiones sin volumen <= {DIAS_SIN_VOLUMEN_MAX:.0%}"]
tabla = pd.DataFrame({
    f"{v / 1e6:g} M$": [(base & (metricas["precio_mediano"] >= p)
                         & (metricas["volumen_dolares_mediano"] >= v)).sum() for p in (1, 5, 10)]
    for v in (500_000, 1_000_000, 5_000_000, 10_000_000)
}, index=["precio >= 1 $", "precio >= 5 $", "precio >= 10 $"])
print(tabla)

# ----------------------------------------------------------
# 5. Las elegibles: volatilidad, sectores y rendimientos extremos
# ----------------------------------------------------------
calidad = pd.read_csv(CARPETA / "calidad_universo.csv", dtype={"ticker": str},
                      keep_default_na=False).set_index("ticker")
elegidas = elegidas.join(calidad[["max_rend_abs", "rendimiento_extremo"]])

print("\n=== ACCIONES ELEGIBLES ===")
print("Volatilidad anual en entrenamiento:")
print(elegidas["volatilidad_anual"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
print("\nPor división SIC:")
print(elegidas["division_sic"].value_counts())
print("\nRendimiento diario > 100 % en algún momento del periodo completo:",
      (elegidas["max_rend_abs"] > 1).sum())
print(elegidas.sort_values("max_rend_abs", ascending=False)
      [["max_rend_abs", "rendimiento_extremo", "precio_mediano", "volumen_dolares_mediano"]].head(15))

# ----------------------------------------------------------
# 6. Guardar
# ----------------------------------------------------------
metricas.to_csv(CARPETA / "metricas_entrenamiento.csv", index_label="ticker")
print("\nGuardado: metricas_entrenamiento.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas: 4838622 | tickers: 2823

=== PRECIOS NO POSITIVOS ===
        open  high  low  close  adj_close
ticker                                   
CBIO       0     0    0      0        725
NFE        0     0    0      0        796
VATE       0     0    0      0        132
              min        max
ticker                      
CBIO   2019-11-04 2022-09-20
NFE    2019-11-04 2022-12-30
VATE   2019-11-04 2020-05-13

=== COCIENTE adj_close / close ===
       cociente_min  cociente_max
count    2,823.0000    2,823.0000
mean         0.9053        0.9975
std          0.1353        0.0086
min         -1.1000        0.7500
1%           0.4430        0.9693
5%           0.6751        0.9888
50%          0.9693        1.0000
95%          1.0000        1.0000
99%          1.0000        1.0000
max          1.0000        1.0000

Los 10 cocientes mínimos más bajos:
        